# Curating Environmental Constraints in S. Tm Context-Specific GEMs

In [1]:
import os

# COBRApy
from cobra.core.configuration import Configuration
from cobra.io import load_model, load_json_model
from cobra.summary import model_summary

In [2]:
# Set the Gurobi license and solver
os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

In [5]:
# load the input S.Tm LT2 GEM
input_model = load_model("STM_v1_0")

In [6]:
# load the stm models generated in `comparing_model_0513.ipynb`
stm1090_model = load_json_model("/home/gmvaz/2026_GEMs/stm_model_test/run_051326/stm1090_imat_restrictions.json")
stm1585_model = load_json_model("/home/gmvaz/2026_GEMs/stm_model_test/run_051326/stm1585_imat_restrictions.json")
stm2575_model = load_json_model("/home/gmvaz/2026_GEMs/stm_model_test/run_051326/stm2575_imat_restrictions.json")

# models are taking >5 mins to load
# (1) check file size on terminal `ls -lh /home/gmvaz/2026_GEMs/stm_model_test/run_051326/*.json`
# (2) check `comparing_models_0513.ipynb` to see if the models were saved immediately after generation
# or after running FBA. If saved after running FBA the models will have the solver states attached to it (making the file larger). 

## What objective functions are available for S. Tm? 

From reading papers on FBA, I was under the impression that people "solve" for appropriate cell objectives and that these were "options" in the GEMs. The S. Tm `input_model' does not have other objective options which leads me to think that you can set any reaction to be the objective. That is to say, you can maximize flux through any desired reaction. 

It would be interesting to set the ATP maintenance reaction as the objective and compare to the biomass reaction results. 

[Here's](http://bigg.ucsd.edu/universal/reactions/BIOMASS_iRR1083_metals) where to see the objective reaction for S. Tm. 

In [7]:
# Let's look at the objective in each model - they should be the same because I did not change them.
# The objective should be to maximize biomass. 
print(input_model.objective)
print(stm1090_model.objective)
print(stm1585_model.objective)
print(stm2575_model.objective)

Maximize
1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Maximize
1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Maximize
1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Maximize
1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4


## What are environmental constraints? 

### Find the minimal media uptake rates in stm_v1_0

The methods section of [(Seif et al., 2018)](https://www.nature.com/articles/s41467-018-06112-5) states that the stm_v1_0 model has uptake rates for in vitro minimal media. Let's first find these. 

**Coding notes** 
I searched the Cobrapy API for `uptake rates` and the cobra.medium module seems promising.

In API, functions are written like this:

  ```
  cobra.medium.find_boundary_types(model: cobra.Model, boundary_type: str, external_compartment: str | None = None) → List[cobra.Reaction]
  ```

and can be read like: 

library: cobra
module: medium
function name: find_boundary_types

parameters, aka inputs (the value passed through a parameter is an argument):
- model
- boundary_type
- external_compartment, optional

output:
- list of cobra Reaction objects




In [6]:
# cobra.medium is a module
# these are the functions in this module: `find_boundary_type`, `find_external_compartment`,
# `is_boundary_type`, and `minimal_media`
